# 🏠 URIS Platform — Colab 完整部署

**一键部署流程（按顺序运行每个 Cell）：**

| Step | 内容 |
|------|------|
| 1️⃣  | 安装依赖 |
| 2️⃣  | 挂载 Drive，clone 项目 |
| 3️⃣  | 部署训练好的 LoRA adapter |
| 4️⃣  | 环境自检 |
| 5️⃣  | 启动 Streamlit + ngrok 公网访问 |

⚠️ 需要：**A100 40GB GPU**（运行时类型 → A100）

## Step 1️⃣ 安装依赖

In [ ]:
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

# 核心依赖
!pip install -q transformers>=4.45.0 peft>=0.13.0 bitsandbytes>=0.43.0
!pip install -q accelerate>=0.34.0 sentencepiece protobuf
!pip install -q qwen-vl-utils

# 平台依赖
!pip install -q streamlit>=1.28.0 pillow numpy opencv-python

# YOLO 目标检测
!pip install -q ultralytics

# ngrok 公网隧道
!pip install -q pyngrok

import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU: {torch.cuda.get_device_name(0)} ({vram:.1f} GB)')
    if vram >= 35:
        print('✅ 显存充足，完整部署可行')
    else:
        print('⚠️  显存偏小，自动启用 4-bit 量化')

## Step 2️⃣ 挂载 Drive + Clone 项目

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

DRIVE_ROOT = '/content/drive/MyDrive/URIS'
os.makedirs(DRIVE_ROOT, exist_ok=True)

# Clone 或更新项目
PROJECT_DIR = '/content/URIS'
if not os.path.exists(PROJECT_DIR):
    !git clone https://github.com/Steven668866/URIS.git {PROJECT_DIR}
else:
    !cd {PROJECT_DIR} && git pull

# 安装项目本身（editable）
!cd {PROJECT_DIR} && pip install -q -e .

print(f'✅ 项目路径: {PROJECT_DIR}')
!ls {PROJECT_DIR}

## Step 3️⃣ 部署 LoRA Adapter

In [ ]:
import os, shutil, json

DRIVE_ROOT = '/content/drive/MyDrive/URIS'
PROJECT_DIR = '/content/URIS'

# adapter 存放的 Drive 路径（你的 zip 解压后放这里）
ADAPTER_IN_DRIVE = f'{DRIVE_ROOT}/final-adapter'
ADAPTER_TARGET = f'{PROJECT_DIR}/Qwen2.5-VL-URIS-Predictive-LoRA'

# 如果 Drive 里有 zip，先解压
zip_path = f'{DRIVE_ROOT}/final-adapter-20260226T023903Z-1-001.zip'
if os.path.exists(zip_path) and not os.path.exists(ADAPTER_IN_DRIVE):
    print('解压 adapter zip...')
    import zipfile
    with zipfile.ZipFile(zip_path, 'r') as zf:
        zf.extractall(DRIVE_ROOT)
    print('✅ 解压完成')

# 复制到项目目录
if os.path.exists(ADAPTER_IN_DRIVE):
    if os.path.exists(ADAPTER_TARGET):
        shutil.rmtree(ADAPTER_TARGET)
    shutil.copytree(ADAPTER_IN_DRIVE, ADAPTER_TARGET)
    # 验证
    cfg = json.load(open(f'{ADAPTER_TARGET}/adapter_config.json'))
    print(f'✅ Adapter 部署成功')
    print(f'   Base: {cfg["base_model_name_or_path"]}')
    print(f'   LoRA: r={cfg["r"]}, alpha={cfg["lora_alpha"]}')
    print(f'   Targets: {cfg["target_modules"]}')
elif os.path.exists(ADAPTER_TARGET):
    cfg = json.load(open(f'{ADAPTER_TARGET}/adapter_config.json'))
    print(f'✅ Adapter 已存在 (r={cfg["r"]})')
else:
    print('❌ 未找到 adapter！')
    print('请把 final-adapter 文件夹上传到 Google Drive 的 URIS/ 目录')

## Step 4️⃣ 环境自检

In [ ]:
import subprocess, sys, os, json

PROJECT_DIR = '/content/URIS'
ok = []
errors = []

# GPU
import torch
if torch.cuda.is_available():
    ok.append(f'GPU: {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_memory/1e9:.1f}GB)')
else:
    errors.append('无 GPU')

# 依赖
for pkg in ['transformers','peft','ultralytics','streamlit','bitsandbytes']:
    try:
        __import__(pkg)
        ok.append(f'{pkg} ✅')
    except ImportError:
        errors.append(f'{pkg} 未安装')

# Qwen 模型类
try:
    from transformers import Qwen2_5_VLForConditionalGeneration
    ok.append('Qwen2_5_VLForConditionalGeneration ✅')
except ImportError:
    errors.append('Qwen2_5_VLForConditionalGeneration 不可用（transformers 版本太旧）')

# Adapter
adapter_path = f'{PROJECT_DIR}/Qwen2.5-VL-URIS-Predictive-LoRA'
if os.path.exists(f'{adapter_path}/adapter_config.json'):
    ok.append('LoRA adapter ✅')
else:
    errors.append('LoRA adapter 不存在')

# URIS 项目结构
for f in ['src/uris_platform/streamlit_app.py',
          'src/uris_platform/services/qwen_adapter.py',
          'src/uris_platform/services/vision_yolo.py']:
    if os.path.exists(f'{PROJECT_DIR}/{f}'):
        ok.append(f'{f.split("/")[-1]} ✅')
    else:
        errors.append(f'{f} 缺失')

print('=== 自检报告 ===')
for x in ok: print(f'  ✅ {x}')
if errors:
    for x in errors: print(f'  ❌ {x}')
    print('\n请解决以上问题后再启动')
else:
    print('\n🎉 全部通过，可以启动平台！')

## Step 5️⃣ 启动 Streamlit + ngrok 公网访问

In [ ]:
# ============================================================
# 填入你的 ngrok authtoken
# 免费注册: https://dashboard.ngrok.com/get-started/your-authtoken
# ============================================================
NGROK_TOKEN = ''  # ← 填入你的 token

import os, subprocess, time, threading
from pyngrok import ngrok, conf

PROJECT_DIR = '/content/URIS'

# 设置 ngrok token
if NGROK_TOKEN:
    conf.get_default().auth_token = NGROK_TOKEN
else:
    print('⚠️  未设置 NGROK_TOKEN，使用匿名隧道（每次重启 URL 会变）')

# 写 streamlit 配置（禁用浏览器自动打开）
os.makedirs(f'{PROJECT_DIR}/.streamlit', exist_ok=True)
with open(f'{PROJECT_DIR}/.streamlit/config.toml', 'w') as f:
    f.write('[server]\nheadless = true\nport = 8501\nenableCORS = false\n')

# 后台启动 Streamlit
env = os.environ.copy()
env['PYTHONPATH'] = f'{PROJECT_DIR}/src'
env['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

proc = subprocess.Popen(
    ['python', '-m', 'streamlit', 'run',
     f'{PROJECT_DIR}/src/uris_platform/streamlit_app.py',
     '--server.port=8501', '--server.headless=true'],
    env=env, cwd=PROJECT_DIR,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT
)

# 等 Streamlit 启动
print('启动 Streamlit...', end='')
for _ in range(30):
    time.sleep(1)
    print('.', end='', flush=True)
    if proc.poll() is not None:
        print('\n❌ Streamlit 启动失败')
        out = proc.stdout.read().decode(errors='replace')
        print(out[-3000:])
        break
else:
    print(' ✅')

# 开 ngrok 隧道
try:
    tunnel = ngrok.connect(8501)
    public_url = tunnel.public_url
    print('\n' + '='*60)
    print('🌐 URIS 平台公网访问地址：')
    print(f'   {public_url}')
    print('='*60)
    print('提示：首次加载 Qwen VLM 约需 1-2 分钟（模型懒加载）')
    print('      YOLO 检测在 Camera 页面 Snapshot 后自动运行')
    print('      Qwen 在你提问时触发推理')
except Exception as e:
    print(f'ngrok 失败: {e}')
    print('直接在 Colab 侧边栏打开端口 8501 也可预览')

## 🔍 可选：查看 Streamlit 日志

In [ ]:
# 查看 Streamlit 输出（调试用）
import select
while True:
    r, _, _ = select.select([proc.stdout], [], [], 0.1)
    if r:
        line = proc.stdout.readline().decode(errors='replace').rstrip()
        if line:
            print(line)
    if proc.poll() is not None:
        print('Streamlit 进程已退出')
        break

## 🧪 可选：离线功能测试（不启动 UI）

In [ ]:
import sys
PROJECT_DIR = '/content/URIS'
sys.path.insert(0, f'{PROJECT_DIR}/src')

from uris_platform.services.qwen_adapter import QwenLiveAdapter

adapter = QwenLiveAdapter(
    adapter_path=f'{PROJECT_DIR}/Qwen2.5-VL-URIS-Predictive-LoRA'
)

print(f'Adapter 状态: {adapter.status}')

# 测试推理（首次加载模型约需 1 分钟）
result = adapter.generate_live_response(
    user_query='帮我看看桌上的水杯',
    scene_summary='书房，检测到 水杯、键盘、鼠标',
    detections=[
        {'label':'水杯','confidence':0.89,'bbox':[100,100,200,200],'center_norm':[0.3,0.4]},
        {'label':'键盘','confidence':0.95,'bbox':[200,300,500,380],'center_norm':[0.6,0.7]},
    ],
    preferences=[],
    recent_turns=[],
    enable_cache=False,
    include_prompt_bundle=False,
)

import json
print('\n=== 用户回复 ===')
print(result.get('user_response', '(无)'))
print('\n=== 行为预测 ===')
aj = result.get('analysis_json', {})
print(f'预测下一步: {aj.get("predicted_next_action", "-")}')
print(f'主动建议: {aj.get("proactive_suggestion", "-")}')
print(f'置信度: {aj.get("confidence", "-")}')
print(f'延迟: {result.get("latency_ms", 0):.0f}ms')